# <b>Task 2.1 - Markout Preparation.</b>

Notebook prepares maker markout data for BTC perpetual futures (Binance).

**Pipeline:**
1. Extract mid_price from LOB (BBO).
2. Extract sign, price, notional from trades.
3. For each trade, compute forward-looking mid at t+τ via asof join.
4. Compute maker PnL in bps: `pnl(τ) = -sign × (mid(τ) - price) / price × 10_000 + 0.5`.
5. Combine all horizons into a single parquet file.

**Horizons:** 30s, 120s, 300s.

## <b> Import Libraries.</b>

In [1]:
import os
import gc
import polars as pl

## <b> Global Variables.</b>

In [2]:
# Params.
HORIZONS            = [30, 120, 300]
REBATE_BPS          = 0.5
CLIPPED_BOUND       = 100_000

In [ ]:
# Paths for BTCUSDT.
PATH_BINANCE_PERP_BTCUSDT_LOB       = r'data\binance_booktickers\perp_btcusdt.parquet'
PATH_BINANCE_PERP_BTCUSDT_LIQUID    = r'data\binance_liquidations\perp_btcusdt.parquet'
PATH_BYBIT_PERP_BTCUSDT_LIQUID      = r'data\bybit_liquidations\btcusdt.parquet'
PATH_BINANCE_PERP_BTCUSDT_TRADES    = r'data\binance_trades\perp_btcusdt.parquet'
PATH_BINANCE_PERP_BTCUSDT_MARKOUT   = r'data\processed\btc_markout.parquet'

# Paths for ETHUSDT.
PATH_BINANCE_PERP_ETHUSDT_LOB       = r'data\binance_booktickers\perp_ethusdt.parquet'
PATH_BINANCE_PERP_ETHUSDT_LIQUID    = r'data\binance_liquidations\perp_ethusdt.parquet'
PATH_BYBIT_PERP_ETHUSDT_LIQUID      = r'data\bybit_liquidations\ethusdt.parquet'
PATH_BINANCE_PERP_ETHUSDT_TRADES    = r'data\binance_trades\perp_ethusdt.parquet'
PATH_BINANCE_PERP_ETHUSDT_MARKOUT   = r'data\processed\eth_markout.parquet'

# All .parquet files.
PARQUET_FILES = {
    "binance_perp_btcusdt_lob": PATH_BINANCE_PERP_BTCUSDT_LOB,
    "binance_perp_btcusdt_liquid": PATH_BINANCE_PERP_BTCUSDT_LIQUID,
    "bybit_perp_btcusdt_liquid": PATH_BYBIT_PERP_BTCUSDT_LIQUID,
    "binance_perp_btcusdt_trades": PATH_BINANCE_PERP_BTCUSDT_TRADES,
    "binance_perp_btcusdt_markout": PATH_BINANCE_PERP_BTCUSDT_MARKOUT,

    "binance_perp_ethusdt_lob": PATH_BINANCE_PERP_ETHUSDT_LOB,
    "binance_perp_ethusdt_liquid": PATH_BINANCE_PERP_ETHUSDT_LIQUID,
    "bybit_perp_ethusdt_liquid": PATH_BYBIT_PERP_ETHUSDT_LIQUID,
    "binance_perp_ethusdt_trades": PATH_BINANCE_PERP_ETHUSDT_TRADES,
    "binance_perp_ethusdt_markout": PATH_BINANCE_PERP_ETHUSDT_MARKOUT
}

In [4]:
lf_lob_btc = pl.scan_parquet(PATH_BINANCE_PERP_BTCUSDT_LOB)
lf_trades_btc = pl.scan_parquet(PATH_BINANCE_PERP_BTCUSDT_TRADES)

## <b>Preprocessing data for PNL calculation.</b>

Extract mid_price from LOB and encode trade side/notional for markout computation.

In [5]:
# calculate mid price.
def preproces_lob(
    lf_lob: pl.LazyFrame,
) -> pl.DataFrame:
    """
    - Preproces LOB Dataframe for PNL calculation.
    
    Return pl.DataFrame with colimns:
    - `timestamp`.
    - `mid_price` (cast to pl.Float32).
    """
    
    df = (
        lf_lob
        .select(
            pl.col("timestamp"),
            ((pl.col("bid_price") + pl.col("ask_price")) / 2).cast(pl.Float32).alias("mid_price")
        )
        .collect(engine="streaming")
    )

    return df


# calculate sign, notional.
def preproces_trades(
    lf_trades: pl.LazyFrame,
) -> pl.DataFrame:
    
    """
    - Preproces Trades Dataframe for PNL calculation.

    Return pl.DataFrame with colimns:
    - `timestamp`.
    - `price` (cast to pl.Float32).
    - `sign` (taker side, cast to pl.Int8).
    - `notional` (aka dollar_amount, cast to pl.Float32).
    """
    
    df_trades = (
        lf_trades
        .select(

            # timestamp.
            pl.col("timestamp"),

            pl.col("price").cast(pl.Float32),

            # code taker side.
            pl.when("buy" == pl.col("side")).then(pl.lit(1, dtype=pl.Int8)).otherwise(pl.lit(-1, dtype=pl.Int8)).alias("sign"),

            # calc notional.
            (pl.col("price") * pl.col("amount")).cast(pl.Float32).alias("notional")
        )
        .collect(engine="streaming")
    )

    return df_trades

In [6]:
# calculations.
print(f"Preprocessing lob for PNL calculation ...")
df_mid = preproces_lob(lf_lob = lf_lob_btc)

print(f"Preprocessing trades for PNL calculation ...")
df_trades = preproces_trades(lf_trades = lf_trades_btc)

print(f"Preprocesed complete.")

# show results.
display(df_trades), display(df_mid)

Preprocessing lob for PNL calculation ...
Preprocessing trades for PNL calculation ...
Preprocesed complete.


timestamp,price,sign,notional
i64,f32,i8,f32
1761955200015000,109557.203125,-1,1095.572021
1761955200052000,109557.296875,1,109.557297
1761955200100000,109557.203125,-1,109.557198
1761955201524000,109557.296875,1,2191.145996
1761955201591000,109557.296875,1,1424.244873
…,…,…,…
1777420797254000,76298.203125,1,9537.275391
1777420797369000,76298.101562,-1,76.298103
1777420797825000,76298.101562,-1,152.596207


timestamp,mid_price
i64,f32
1761955200491000,109557.25
1761955201433000,109557.25
1761955201484000,109557.25
1761955201536000,109557.25
1761955201640000,109557.25
…,…
1777420798877000,76298.148438
1777420798929000,76298.148438
1777420799397000,76298.148438


(None, None)

## <b>Calculate PNL.</b>

For each horizon τ ∈ {30s, 120s, 300s}: asof join trade with mid_price at t+τ, compute maker PnL in bps. Combine all horizons into single parquet file.

Formula: `pnl(τ) = -sign × (mid(τ) - price) / price × 10_000 + 0.5`

In [7]:
def calculate_pnl(
    df_trades: pl.DataFrame,
    df_mid: pl.DataFrame
):
    for tau in HORIZONS:
        df_result = (
            df_trades
            .with_columns((pl.col("timestamp") + tau * 1_000_000).alias("lookup_ts"))
            .join_asof(df_mid, left_on="lookup_ts", right_on="timestamp", strategy="backward")
            .drop("lookup_ts", "timestamp_right")
            .with_columns(
                (-pl.col("sign") * (pl.col("mid_price") - pl.col("price")) / pl.col("price") * 10_000 + REBATE_BPS).cast(pl.Float32).alias("pnl_bps"),
                pl.col("notional").clip(upper_bound=CLIPPED_BOUND).cast(pl.Float32).alias("weight")
            )
        )
        df_result.write_parquet(f"data/processed/btc_markout_{tau}s.parquet")
        del df_result
        gc.collect()

In [8]:
# calculate PNL.
calculate_pnl(
    df_trades   = df_trades,
    df_mid      = df_mid
)

# collect PNL result in single a file.
df_pnl = pl.read_parquet("data/processed/btc_markout_30s.parquet").rename({"pnl_bps": "pnl_30s"}).drop("mid_price")
df_pnl = df_pnl.with_columns(
    pl.read_parquet("data/processed/btc_markout_120s.parquet")["pnl_bps"].alias("pnl_120s"),
    pl.read_parquet("data/processed/btc_markout_300s.parquet")["pnl_bps"].alias("pnl_300s")
)

# show results.
display(df_pnl)

timestamp,price,sign,notional,pnl_30s,weight,pnl_120s,pnl_300s
i64,f32,i8,f32,f32,f32,f32,f32
1761955200015000,109557.203125,-1,1095.572021,3.032923,1095.572021,-4.059547,-13.388293
1761955200052000,109557.296875,1,109.557297,-2.024364,109.557297,5.0681,14.396838
1761955200100000,109557.203125,-1,109.557198,3.032923,109.557198,-4.059547,-13.388293
1761955201524000,109557.296875,1,2191.145996,-2.024364,2191.145996,5.716306,14.396838
1761955201591000,109557.296875,1,1424.244873,-2.024364,1424.244873,5.716306,14.396838
…,…,…,…,…,…,…,…
1777420797254000,76298.203125,1,9537.275391,0.507168,9537.275391,0.507168,0.507168
1777420797369000,76298.101562,-1,76.298103,0.506144,76.298103,0.506144,0.506144
1777420797825000,76298.101562,-1,152.596207,0.506144,152.596207,0.506144,0.506144


In [9]:
# deleate over files and write df_pnl. 
os.remove("data/processed/btc_markout_30s.parquet")
os.remove("data/processed/btc_markout_120s.parquet")
os.remove("data/processed/btc_markout_300s.parquet")

df_pnl.write_parquet(f"data/processed/btc_markout.parquet")